# Челленджи недели: память, потоки, процессы и asyncio

**Цель:** проверить, что концепты concurrency работают вместе и применяются к реальным задачам — гонкам данных, параллельной обработке, асинхронному I/O.

**Как работать:**
- Часть задач сопровождается ячейкой «Объясни своими словами» — там нужно не только написать код, но и сформулировать, почему он работает.
- Задачи решаются на 5-30 строк кода. На W4 разрешены все Python-фичи: `def`, `class`, `async def`, `threading`, `multiprocessing`, `asyncio`.
- Сначала попробуй сам, без подглядываний. Если застрял на 10+ минут — открой solution-версию.

**Где решать.** Большинство заданий — прямо в ячейках ноутбука. Исключение — задание 2 (`multiprocessing.Pool`). Его нужно решать в отдельном `.py`-файле, в задании увидишь блок:

> **→ Перейди в терминал.** `python scripts/task_02_pool_practice.py`

Причина — `Pool` в Jupyter на Windows не работает (дочерние процессы не видят функции, объявленные в ячейках ноутбука). Подробности — в `scripts/README.md`.

**Setup для Jupyter.** Asyncio-задачи зовут `asyncio.run(...)` внутри ячеек. Jupyter уже крутит свой event loop, поэтому нужен `nest_asyncio.apply()` ниже. Если пакет не установлен — `pip install nest_asyncio`.


In [1]:
# Технический setup для Jupyter — это НЕ материал недели, а среда исполнения.
# Одна строка нужна, чтобы ноутбук вообще запустился; ничего здесь учить не надо.
#
# nest_asyncio.apply() — Jupyter уже крутит свой event loop в фоне, и
# `asyncio.run(...)` внутри ячейки иначе падает с RuntimeError. Этот патч
# разрешает повторный запуск loop'а поверх существующего. В обычном `.py`
# скрипте такая строка не нужна.
import nest_asyncio
nest_asyncio.apply()
print("setup OK")


setup OK


## Задание 1: Потокобезопасный счётчик через `Lock`

Реализуй класс `SafeCounter` с методом `increment()`, который безопасно увеличивает внутренний счётчик из нескольких потоков. Внутри используй `threading.Lock` через `with self._lock:`.

Затем запусти 10 потоков, каждый из которых делает 10 000 инкрементов. Финальное значение должно быть ровно `100_000` — это и есть проверка, что race condition нет.

Подсказка: без `Lock` некоторые инкременты потеряются (`counter += 1` это три шага: read / add / write).

In [2]:
import threading

class SafeCounter:
    def __init__(self):
        self._value = 0
        self._lock = threading.Lock()

    def increment(self):
        with self._lock:
            self._value += 1

    @property
    def value(self):
        return self._value

def worker(counter, n):
    for _ in range(n):
        counter.increment()

counter = SafeCounter()
threads = [
    threading.Thread(target=worker, args=(counter, 10_000))
    for _ in range(10)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter.value)   # 100000 — все инкременты сохранены


100000


**Объясни своими словами:** что произойдёт, если убрать `with self._lock:` из метода `increment`?

Без `Lock` инкременты теряются. `self._value += 1` на уровне байткода это три отдельные операции: загрузить текущее значение в регистр, прибавить 1, записать обратно. GIL переключает потоки между байткодами, поэтому два потока могут одновременно прочитать значение `42`, оба прибавить 1, оба записать `43` — один инкремент потерян. На 10 потоков × 10 000 итераций без `Lock` финальное значение будет где-то `60K-95K` (зависит от запуска), а не `100_000`. `Lock` гарантирует, что критическая секция (`read → add → write`) выполняется атомарно: пока один поток внутри `with`, остальные ждут на `_lock.acquire()`.

## Задание 2: CPU-bound — параллельная сумма квадратов через `Pool`

Реализуй функцию `heavy_sum_of_squares(n)`, которая считает `sum(i*i for i in range(n))`. На больших `n` (несколько миллионов) это чисто CPU-задача.

Затем сравни два способа:

1. **Последовательно**: прогон функции на 4 значениях `n` через обычный `for`.
2. **Параллельно через `multiprocessing.Pool(4)`**: те же 4 значения через `pool.map`.

Замерь время через `time.perf_counter()` и выведи speedup (`seq_time / pool_time`). На 4-ядерной машине ожидаем ~3-4x ускорения.

> **→ Перейди в терминал.** Задание решается в отдельном `.py`-файле — шаблон с `TODO` уже лежит рядом. Открой `scripts/task_02_pool_practice.py` в редакторе, заполни блоки `TODO` и запусти из папки `notebooks/`:
>
> ```bash
> python scripts/task_02_pool_practice.py
> ```
>
> Когда увидишь speedup и `результаты совпали: True` — сравни своё решение с `scripts/task_02_pool_solution.py` и возвращайся к следующему заданию ноутбука.

**Почему отдельный скрипт.** В Jupyter `multiprocessing.Pool` кросс-платформенно не работает: на Windows дочерние процессы не видят функции, объявленные в ячейках ноутбука. Поэтому правильный паттерн для `Pool` — `.py`-файл с гвардом `if __name__ == "__main__":`.

**Объясни своими словами:** почему здесь нужен `multiprocessing.Pool`, а не `ThreadPoolExecutor`?

Задача чисто CPU-bound: внутри функции `heavy_sum_of_squares` нет ни сетевых, ни дисковых вызовов — только арифметика на байткоде Python. На байткоде GIL держит ровно один поток в активной фазе; 4 потока на CPU-задаче работают столько же, сколько 1 поток (общее время не меняется, просто переключаются между собой). `multiprocessing.Pool` создаёт 4 отдельных процесса, у каждого свой Python-интерпретатор и свой GIL — они работают на 4 ядрах одновременно. Цена — данные между процессами передаются через `pickle`-сериализацию, и сам старт процесса дороже старта потока. Для CPU-задачи это окупается: `Pool` на 4 миллиона итераций × 4 задачи даёт ускорение ~3-4x; `ThreadPoolExecutor` — близко к 1x.

## Задание 3: Параллельные HTTP-запросы через `asyncio.gather`

Симулируй параллельные HTTP-запросы. Без реальной сети — используем `asyncio.sleep(latency)` как заглушку.

- Напиши `async def fetch(url, latency)` — печатает «start url», засыпает на `latency` секунд через `await asyncio.sleep(latency)`, печатает «done url» и возвращает строку `f"<{url}>"`.
- Напиши `async def main()` — берёт 4 URL'а с разными latency (0.3, 0.2, 0.4, 0.1 с), запускает `asyncio.gather(...)` параллельно, замеряет время, печатает результаты и общее время.
- Запусти через `asyncio.run(main())`. Общее время должно быть ≈0.4с (равно самой медленной), а не 1.0с (сумма).

In [3]:
import asyncio
import time

async def fetch(url, latency):
    print(f"  start  {url}")
    await asyncio.sleep(latency)
    print(f"  done   {url}")
    return f"<{url}>"

async def main():
    urls = [
        ("api/users",    0.3),
        ("api/orders",   0.2),
        ("api/products", 0.4),
        ("api/health",   0.1),
    ]
    start = time.perf_counter()
    results = await asyncio.gather(*(fetch(u, lat) for u, lat in urls))
    elapsed = time.perf_counter() - start
    print()
    print(f"результаты: {results}")
    print(f"общее время: {elapsed:.2f}s — близко к 0.4 (max latency), не 1.0 (sum)")

asyncio.run(main())


  start  api/users
  start  api/orders
  start  api/products
  start  api/health
  done   api/health


  done   api/orders
  done   api/users
  done   api/products

результаты: ['<api/users>', '<api/orders>', '<api/products>', '<api/health>']
общее время: 0.40s — близко к 0.4 (max latency), не 1.0 (sum)


## Задание 4: Rate-limiting через `asyncio.Semaphore`

Бывает: API позволяет максимум 3 одновременных запроса; больше — даёт `429 Too Many Requests`. Решение — `asyncio.Semaphore(3)`: семафор пропускает не больше 3 параллельно, остальные ждут освобождения слота.

- Напиши `async def fetch(sem, idx)` — внутри `async with sem:` печатает «start idx», засыпает на `0.2` с, печатает «done idx».
- В `main()` запусти 10 задач через `asyncio.gather`. Семафор должен пропускать только 3 одновременно — посмотри по принту, как они идут волнами по 3.

Замерь общее время — для 10 задач × 0.2с при лимите 3 ожидаем `~ceil(10/3) * 0.2 = 0.8с`.

In [4]:
import asyncio
import time

async def fetch(sem, idx):
    async with sem:
        print(f"  start  task {idx}")
        await asyncio.sleep(0.2)
        print(f"  done   task {idx}")
        return idx

async def main():
    sem = asyncio.Semaphore(3)
    start = time.perf_counter()
    results = await asyncio.gather(*(fetch(sem, i) for i in range(10)))
    elapsed = time.perf_counter() - start
    print()
    print(f"результаты: {results}")
    print(f"общее время: {elapsed:.2f}s — близко к 0.8 (4 волны по 3 + хвост)")

asyncio.run(main())


  start  task 0
  start  task 1
  start  task 2
  done   task 0
  done   task 1
  done   task 2


  start  task 3
  start  task 4
  start  task 5


  done   task 3
  done   task 4
  done   task 5
  start  task 6
  start  task 7
  start  task 8


  done   task 6
  done   task 7
  done   task 8
  start  task 9


  done   task 9

результаты: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
общее время: 0.81s — близко к 0.8 (4 волны по 3 + хвост)


## Задание 5: Блокирующий код через `asyncio.to_thread`

Бывает: нужно из `async def` позвать обычную блокирующую функцию (нет async-версии библиотеки или legacy-код). Если позвать напрямую — она заморозит event loop на всё время выполнения. Решение — `asyncio.to_thread(func, *args)`: выносит вызов в отдельный поток, event loop остаётся свободным.

- Напиши блокирующую `def blocking_compute(name, seconds)` — `time.sleep(seconds)` + возвращает строку.
- В `async def main()` запусти 3 таких вызова **параллельно** через `asyncio.gather(*(asyncio.to_thread(blocking_compute, name, sec) for ...))`.
- Замерь время — несмотря на то что `blocking_compute` блокирующая, через `to_thread` три вызова идут параллельно (общее время ~max).

In [5]:
import asyncio
import time

def blocking_compute(name, seconds):
    print(f"  start  {name}")
    time.sleep(seconds)   # БЛОКИРУЮЩИЙ вызов — внутри корутины напрямую нельзя
    print(f"  done   {name}")
    return f"<{name}>"

async def main():
    start = time.perf_counter()
    results = await asyncio.gather(
        asyncio.to_thread(blocking_compute, "alpha", 0.3),
        asyncio.to_thread(blocking_compute, "beta",  0.2),
        asyncio.to_thread(blocking_compute, "gamma", 0.4),
    )
    elapsed = time.perf_counter() - start
    print()
    print(f"результаты: {results}")
    print(f"общее время: {elapsed:.2f}s — близко к 0.4 (max), не 0.9 (sum)")

asyncio.run(main())


  start  alpha
  start  beta
  start  gamma


  done   beta
  done   alpha
  done   gamma

результаты: ['<alpha>', '<beta>', '<gamma>']
общее время: 0.40s — близко к 0.4 (max), не 0.9 (sum)


## Задание 6: Асинхронный генератор

Асинхронный генератор — это функция, объявленная через `async def`, которая использует `yield`. По ней итерируются через `async for`.

- Реализуй `async def stream_events(n)` — асинхронный генератор, который выдаёт `n` событий: между ними `await asyncio.sleep(0.05)` (имитация прихода события из сети).
- Каждое событие — словарь `{"id": i, "timestamp": time.perf_counter()}`.
- В `main()` итерируй через `async for event in stream_events(5):` и печатай каждое событие.

Подсказка: внутри `async def` с `yield` нельзя писать `return value` — это синтаксическая ошибка для async-генераторов. Только `yield`.

In [6]:
import asyncio
import time

async def stream_events(n):
    for i in range(n):
        await asyncio.sleep(0.05)
        yield {"id": i, "timestamp": round(time.perf_counter(), 3)}

async def main():
    async for event in stream_events(5):
        print("  event:", event)

asyncio.run(main())


  event: {'id': 0, 'timestamp': 2.394}
  event: {'id': 1, 'timestamp': 2.444}
  event: {'id': 2, 'timestamp': 2.496}
  event: {'id': 3, 'timestamp': 2.547}


  event: {'id': 4, 'timestamp': 2.598}


## Задание 7: Циклическая ссылка и сборщик мусора

Создай два объекта, ссылающихся друг на друга — циклическая ссылка. Покажи, что после `del` обычных переменных счётчик ссылок не падает до 0 (объекты живы), и только `gc.collect()` их освобождает.

- Сделай простой класс `Node` с атрибутом `partner` и `__del__`-методом, который печатает «удалили <name>» (так увидим момент освобождения).
- Создай `a = Node("A")`, `b = Node("B")`, свяжи их через `a.partner = b; b.partner = a`.
- Удали локальные ссылки: `del a; del b`. До `gc.collect()` `__del__` не вызывался — это видно по отсутствию принта.
- Вызови `gc.collect()` — оба `__del__` напечатают своё сообщение.

In [7]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None

    def __del__(self):
        print(f"  удалили Node({self.name})")

# Делаем циклическую ссылку
a = Node("A")
b = Node("B")
a.partner = b
b.partner = a

print("удаляем локальные ссылки a и b...")
del a
del b
print("...прошло; __del__ ещё не вызывался — счётчик не упал до 0")

print("вызываем gc.collect()...")
collected = gc.collect()
print(f"...gc собрал {collected} объектов")


удаляем локальные ссылки a и b...
...прошло; __del__ ещё не вызывался — счётчик не упал до 0
вызываем gc.collect()...
  удалили Node(A)
  удалили Node(B)
...gc собрал 4 объектов


**Объясни своими словами:** почему `del a; del b` сами по себе не освободили объекты, а `gc.collect()` смог?

У каждого объекта свой счётчик ссылок. Когда мы пишем `a.partner = b`, у `b` появляется вторая ссылка (одна из локальной `b`, вторая из атрибута `a.partner`). Симметрично — у `a` две ссылки. После `del a` локальной `a` нет, но `b.partner` всё ещё держит её — счётчик `1`. После `del b` локальной `b` тоже нет, но `a.partner` держит её — счётчик `1`. Получается замкнутый круг: `A` живёт из-за ссылки в `B`, `B` живёт из-за ссылки в `A`. Счётчики у обоих `1`, но снаружи на них никто не ссылается — это и есть **циклическая ссылка**. Reference counting сам не справляется. `gc.collect()` — отдельный механизм, который ходит по всем «подозрительным» объектам, строит граф ссылок и находит замкнутые циклы, недостижимые из корней программы. Найдя такой цикл, он разрывает связи и зовёт `__del__` на обоих объектах.

# Готово

Ты только что прошёл задачи на пересечении модели памяти, потоков, процессов и asyncio. Race condition с `Lock`, параллельная обработка CPU-задач через `Pool`, асинхронные HTTP-вызовы через `gather`, rate-limiting через `Semaphore`, блокирующий код через `to_thread`, async-генераторы и циклические ссылки — это набор паттернов, которые встречаются в backend и ML-инфраструктуре каждый день.

На следующей неделе мы переключимся на специфику вашего трека — дообучение языковых моделей (NLP fine-tuning). Concurrency-инструменты из этой недели уйдут в фон — они там везде, но больше не будут главной темой.
